In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
import os
import logging
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from wsi.nets.vision_transformer import vit_small
from wsi.wsi import *
from wsi.utils import *
from wsi.nets.dasmil import *

from torch_geometric.data import Data
from sklearn.model_selection import train_test_split


In [ ]:
# def get_data(path,device):
#     # level2--10
#     # level3--20
#     data = torch.load(path)
#     features = data.x[data.level==3].to(device)
#     edges = data.edge_index_3.to(device)
#     label = data.y.to(device)
#     return features,edges,label

def get_data(path,device):
    data = torch.load(path)
    features = data["features"].to(device)
    label = torch.tensor([data["label"]]).to(device)
    rects = {(x,y):i for i,(x,y) in enumerate(data["rects"])}
    edges = []
    s = data["patch_size"]
    for i,(x,y) in enumerate(rects):
        for x_i,y_i in [(x+s,y-s),(x+s,y),(x+s,y+s),(x,y-s),(x,y+s),(x-s,y-s),(x-s,y),(x-s,y+s)]:
            if (x_i,y_i) in rects:
                edges.append([i,rects[(x_i,y_i)]])
    edges = torch.tensor(edges).T.to(device)
    return features,edges,label


# x,e,_ = get_data("../../BRACS/pkl_data/20xbags/BRACS_1003667_12148_2.pth","cpu")
# x,e,_ = get_data("../../camGraph_23/processed/test/data_4.pt","cpu")
# x.shape,e.shape


In [ ]:
gnn_layer_by_name = {
    "GCN": geom_nn.GCNConv,
    "GATV2": geom_nn.GATv2Conv,
    "GAT": geom_nn.GATConv,
    "GraphConv": geom_nn.GraphConv
}

class GNNModel(nn.Module):
    def __init__(self, c_in, c_hidden, c_out, num_layers=2, layer_name="GAT", dp_rate=0.1, heads=3):
        super().__init__()
        gnn_layer = gnn_layer_by_name[layer_name]
        layers = []
        in_channels, out_channels = c_in, c_hidden
        for l_idx in range(num_layers-1):
            layers += [
                gnn_layer(in_channels=in_channels,out_channels=out_channels),
                nn.ReLU(inplace=True),
                nn.Dropout(dp_rate)
            ]
            in_channels = c_hidden

        layers += [gnn_layer(in_channels=in_channels,out_channels=c_out,heads=heads,concat=False,dropout=dp_rate) if "GAT" in layer_name else gnn_layer(in_channels=in_channels,out_channels=c_out)]

        self.layers = nn.ModuleList(layers)

    def forward(self, x, edge_index):
        # GNN 对应的层
        for l in self.layers:
            if isinstance(l, geom_nn.MessagePassing):
                x = l(x, edge_index)
            else:
                x = l(x)
        return x


class FCLayer(nn.Module):
    def __init__(self, in_size, out_size=1):
        super(FCLayer, self).__init__()
        self.fc = nn.Sequential(nn.Linear(in_size, out_size))

    def forward(self, feats):
        x = self.fc(feats)
        return feats, x


# Bag-level classifier class
class BClassifier(nn.Module):
    def __init__(self, input_size, output_class, dropout_v=0.0, nonlinear=True):  # K, L, N
        super(BClassifier, self).__init__()
        if nonlinear:
            # Non-linear transformation
            self.lin = nn.Sequential(
                nn.Linear(input_size, input_size), nn.ReLU())
            self.q = nn.Sequential(nn.Linear(input_size, 128), nn.Tanh())
        else:
            # Identity transformation
            self.lin = nn.Identity()
            self.q = nn.Linear(input_size, 128)
        self.v = nn.Sequential(
            nn.Dropout(dropout_v),
            nn.Linear(input_size, input_size)
        )

        # 1D convolutional layer that can handle multiple class (including binary)
        self.fcc = nn.Sequential(
            nn.Conv1d(output_class, output_class, kernel_size=input_size),
            )

    def forward(self, feats, c):
        # N x K, N x C
        device = feats.device
        feats = self.lin(feats)
        V = self.v(feats)  # N x V, unsorted
        Q = self.q(feats).view(feats.shape[0], -1)  # N x Q, unsorted

        # Handle multiple classes without for loop
        # Sort class scores along the instance dimension, m_indices in shape N x C
        _, m_indices = torch.sort(c, 0, descending=True)
        # Select critical instances, m_feats in shape C x K
        m_feats = torch.index_select(feats, dim=0, index=m_indices[0, :])
        # Compute queries of critical instances, q_max in shape C x Q
        q_max = self.q(m_feats)
        # Compute inner product of Q to each entry of q_max, A in shape N x C, each column contains unnormalized attention scores
        A = torch.mm(Q, q_max.transpose(0, 1))
        # Normalize attention scores, A in shape N x C,
        A = F.softmax(A / torch.sqrt(torch.tensor(Q.shape[1], dtype=torch.float32, device=device)), 0)
        # Compute bag representation, B in shape C x V
        B = torch.mm(A.transpose(0, 1), V)

        B = B.view(1, B.shape[0], B.shape[1])  # 1 x C x V
        C = self.fcc(B)  # 1 x C x 1
        C = C.view(1, -1)
        return C, A, B
    
# 常规自注意力
class BClassifier_noselect(nn.Module):
    def __init__(self, input_size, output_class, dropout_v=0.0):  # K, L, N
        super(BClassifier_noselect, self).__init__()
        # Non-linear transformation
        self.lin = nn.Sequential(
            nn.Linear(input_size, input_size), nn.ReLU())
        self.q = nn.Sequential(nn.Dropout(dropout_v),nn.Linear(input_size, 128))
        self.k = nn.Sequential(nn.Dropout(dropout_v),nn.Linear(input_size, 128))
        self.v = nn.Sequential(nn.Dropout(dropout_v),nn.Linear(input_size, input_size))

        # 1D convolutional layer that can handle multiple class (including binary)
        self.fcc = nn.Sequential(
            nn.Linear(input_size,output_class)
            )

    def forward(self, feats, c):
        # N x K, N x C
        device = feats.device
        feats = self.lin(feats)
        Q = self.q(feats).view(feats.shape[0], -1)  # N x Q, unsorted
        K = self.k(feats)  # N x Q, unsorted
        V = self.v(feats)  # N x V, unsorted

        A = torch.mm(Q, K.transpose(0, 1)) # N x N
        A = F.softmax(A / torch.sqrt(torch.tensor(Q.shape[1], dtype=torch.float32, device=device)), 0)
        B = torch.mm(A.transpose(0, 1), V) # N x V

        B = B.view(1, B.shape[0], B.shape[1])  # 1 x N x V
        C = self.fcc(B[:,0,:])  # 1 x C
        return C, A, B

# MILNet module for multiple instance learning.
class MILNet(nn.Module):
    def __init__(self, 
                 i_classifier, # i_classifier (nn.Module): Instance-level classifier 
                 b_classifier # Bag-level classifier.
                 ):
        super(MILNet, self).__init__()
        self.i_classifier = i_classifier
        self.b_classifier = b_classifier

    def forward(self, x):
        feats, classes = self.i_classifier(x)
        prediction_bag, A, B = self.b_classifier(feats, classes)

        return classes, prediction_bag, A, B
    


class MyModel(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil
    def forward(self,features,edges):
        features = self.gnn(features,edges)
        prediction_i, prediction_b, attscore, feature_b = self.mil(features)
        return prediction_i, prediction_b, attscore, feature_b


In [ ]:
from pathlib import Path
from datetime import datetime
import json

script = Path(
    "/data/code/WSI/WSI_pretrain/"
    "retrain_baselines_patience10_on_BRACS.py"
)

if not script.is_file():
    raise FileNotFoundError(script)

required = [
    "get_data",
    "GNNModel",
    "FCLayer",
    "BClassifier",
    "MILNet",
    "MyModel",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        f"请先执行原模型类或函数的定义：{missing}"
    )

# 沿用本轮BRACS公共划分，不重新随机划分
original_root = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/compare/BRACS/"
)
split_source = original_root / "patience10_split_on_BRACS.json"

if not split_source.is_file():
    raise FileNotFoundError(split_source)

split = json.loads(split_source.read_text(encoding="utf-8"))
assert len(split["train"]) == 99
assert len(split["val"]) == 25
assert set(split["train"]).isdisjoint(split["val"])

# 新建本次重训目录
ours_bracs_rerun_root = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/temp/"
) / (
    "BRACS_Ours_Rerun_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
ours_bracs_rerun_root.mkdir(parents=True, exist_ok=False)

# 原样复制公共划分
split_target = (
    ours_bracs_rerun_root / "patience10_split_on_BRACS.json"
)
split_target.write_bytes(split_source.read_bytes())

print("沿用划分：", split_source, flush=True)
print("本次实验目录：", ours_bracs_rerun_root, flush=True)
print("开始 BRACS Ours 重训", flush=True)

get_ipython().run_line_magic(
    "run",
    f'-i "{script}" '
    '--dataset BRACS --models Ours '
    '--data-path "/data/database/BRACS/pkl_data/old/train/" '
    f'--save-root "{ours_bracs_rerun_root}"'
)

result_dir = ours_bracs_rerun_root / "Ours/Ours_patience10"
summary_path = result_dir / "training_summary.json"
weight_path = result_dir / "model.pth"

if summary_path.is_file() and weight_path.is_file():
    print("\n训练完成")
    print("最佳权重：", weight_path)
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("\n未找到完整训练结果，请检查上方报错。")

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# 若已重启内核，填写本次重训目录；否则留空
rerun_folder = ""
# 示例：
# rerun_folder = (
#     "/data/code/WSI/WSI_pretrain/model/MIL/temp/"
#     "BRACS_Ours_Rerun_实际时间戳"
# )

if rerun_folder:
    experiment_dir = Path(rerun_folder)
elif "ours_bracs_rerun_root" in globals():
    experiment_dir = Path(ours_bracs_rerun_root)
else:
    raise ValueError("请在 rerun_folder 中填写本次重训目录。")

assert experiment_dir.name.startswith("BRACS_Ours_Rerun_"), (
    f"请检查本次重训目录：{experiment_dir}"
)

run_dir = experiment_dir / "Ours/Ours_patience10"
test_dir = Path("/data/database/BRACS/pkl_data/old/test/")
weight_path = run_dir / "model.pth"

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert config["model"] == "Ours"
assert config["dataset"] == "BRACS"
assert config["class_num"] == 4

# 请先执行原模型类和get_data的定义
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("本次测试权重：", weight_path.resolve())

model = torch.load(
    weight_path,
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 31, (
    f"预期31张测试切片，实际为{len(test_files)}张"
)

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="BRACS Ours Rerun Test"):
        features, edges, label = get_data(str(path), device)

        # 使用与训练一致的空间边，包分支输出分类
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 4), (
            f"{path.name} 输出形状异常：{tuple(bag_logits.shape)}"
        )
        assert torch.isfinite(bag_logits).all(), (
            f"{path.name} 输出包含NaN或Inf"
        )

        true_label = int(label.item())
        prediction = int(bag_logits.argmax(dim=1).item())
        assert true_label in (0, 1, 2, 3)

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
        })

distribution = [labels.count(i) for i in range(4)]
assert distribution == [12, 6, 7, 6], (
    f"测试集标签分布不一致：{distribution}"
)

class_labels = [0, 1, 2, 3]

result = {
    "model": "BRACS_Ours_Rerun_patience10",
    "run_id": experiment_dir.name,
    "weight_path": str(weight_path.resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "macro",
    "precision": float(precision_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "confusion_matrix_labels_0_1_2_3": confusion_matrix(
        labels, predictions, labels=class_labels
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序为0、1、2、3")
print(np.array(result["confusion_matrix_labels_0_1_2_3"]))

# 保存到本次temp实验目录
# 重复运行会更新测试JSON，不修改权重
with (run_dir / "test_metrics.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

with (run_dir / "test_predictions.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

# 追加测试成绩，保留训练日志
log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1, 2, 3]:\n"
        + str(result["confusion_matrix_labels_0_1_2_3"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果已保存至：", run_dir.resolve())
print("测试成绩已写入日志：", log_path.resolve())

In [ ]:
device = torch.device("cuda:0")
class_num = 2

# GAT1
model_gnn = GNNModel(384,256,256,num_layers=1).to(device)
# GAT2
# model_gnn = GNNModel(384,256,256,num_layers=2).to(device)
# GAT3
# model_gnn = GNNModel(384,256,256,num_layers=3).to(device)

# GCN1
# model_gnn = GNNModel(384,256,256,num_layers=1,layer_name="GCN").to(device)
# GCN2
# model_gnn = GNNModel(384,256,256,num_layers=2,layer_name="GCN").to(device)
# GCN3
# model_gnn = GNNModel(384,256,256,num_layers=3,layer_name="GCN").to(device)

# 重要实例
model_bclassifier = BClassifier(256,class_num).to(device)
# 常规自注意力
# model_bclassifier = BClassifier_noselect(256,class_num).to(device)

model_iclassifier = FCLayer(256,class_num).to(device)
model_mil = MILNet(model_iclassifier,model_bclassifier)
model = MyModel(model_gnn,model_mil)


loss_func = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(),3e-4)



treain_data_path = "../../CAMELYON16/pkl_data/training/20xbags/"
train_data_files = np.array(os.listdir(treain_data_path))
train_data_files,val_data_files = train_test_split(train_data_files,test_size=0.2,random_state=42)


save_path = "./model/MIL/temp/ours/"
if not os.path.exists(save_path):
    os.mkdir(save_path)

logger = logging.getLogger('logger')
logger.setLevel(logging.INFO)
logger.handlers=[]
fh = logging.FileHandler(os.path.join(save_path,'log.log'),"w")
fh.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(message)s')
fh.setFormatter(formatter)
logger.addHandler(fh)


loss_min = np.inf
# patience = 10
patience = 5
e_count = patience
for e in range(100):
    model.train()
    train_loss = []
    for f in tqdm(train_data_files):
        features,edges,label = get_data(os.path.join(treain_data_path,f),device)
        if len(edges)<1:
            continue
        
        prediction_i, prediction_b, attscore, feature_b = model(features,edges)
        
        optim.zero_grad()
        loss = loss_func(prediction_b,label)
        loss.backward()
        optim.step()
        train_loss.append(loss.item())


    model.eval()
    val_loss = []
    for f in tqdm(val_data_files):
        with torch.no_grad():
            features,edges,label = get_data(os.path.join(treain_data_path,f),device)

            prediction_i, prediction_b, attscore, feature_b = model(features,edges)        
            loss = loss_func(prediction_b,label)
            val_loss.append(loss.item())



    train_loss,val_loss = np.mean(train_loss),np.mean(val_loss)
    info = f"{e}/{e_count}\ttrain:{train_loss}\tval:{val_loss}\t"
    if val_loss<loss_min:
        e_count = patience
        loss_min = val_loss
        info+="saved"
        torch.save(model,os.path.join(save_path,"model.pth"))

    e_count-=1
    if e_count<1:
        break

    logger.info(info)
    print(info)



In [ ]:
import os
import json
import logging

import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# 先执行原Notebook中以下类及函数的定义：
# GNNModel、FCLayer、BClassifier、MILNet、MyModel、get_data

device = torch.device("cuda:0")
class_num = 2

train_data_path = (
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)

from sklearn.model_selection import train_test_split

train_data_files = np.array(os.listdir(train_data_path))

train_data_files, val_data_files = train_test_split(
    train_data_files,
    test_size=0.2,
    random_state=42,
)

train_files = list(train_data_files)
val_files = list(val_data_files)

assert len(train_files) == 216
assert len(val_files) == 54
assert set(train_files).isdisjoint(val_files)

for filename in train_files + val_files:
    path = os.path.join(train_data_path, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

save_path = "./model/MIL/temp/ours/"
os.makedirs(save_path, exist_ok=True)

# 防止误覆盖已经完成的本次实验
for filename in ("model.pth", "log.log", "split.json"):
    path = os.path.join(save_path, filename)
    if os.path.exists(path):
        raise FileExistsError(f"已有实验文件：{path}")

with open(
    os.path.join(save_path, "split.json"),
    "w", encoding="utf-8",
) as f:
    json.dump(
        {"train": train_files, "val": val_files},
        f, ensure_ascii=False, indent=2,
    )

# 重新初始化本文架构
model_gnn = GNNModel(
    384, 256, 256,
    num_layers=1,
).to(device)

model_iclassifier = FCLayer(
    256, class_num
).to(device)

model_bclassifier = BClassifier(
    256, class_num
).to(device)

model_mil = MILNet(
    model_iclassifier,
    model_bclassifier,
)

model = MyModel(
    model_gnn,
    model_mil,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
)

logger = logging.getLogger("ours_patience5")
logger.setLevel(logging.INFO)
logger.propagate = False

for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()

handler = logging.FileHandler(
    os.path.join(save_path, "log.log"),
    mode="w",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)

logger.info(
    "CAMELYON16 | Ours | single-layer GAT | "
    "original spatial edges | "
    "0.5 bag CE + 0.5 max-instance CE | "
    "Adam lr=3e-4 | max_epochs=100 | patience=5 | "
    "historical countdown: decrement after every epoch"
)

best_val_loss = np.inf
best_epoch = None
patience = 5
remaining = patience

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files,
            desc=f"Epoch {epoch} Train",
        ):
            features, edges, label = get_data(
                os.path.join(train_data_path, filename),
                device,
            )

            # 使用原始空间边，不替换为仅自环
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            target = label.long().reshape(-1)

            bag_loss = F.cross_entropy(
                bag_logits, target
            )

            max_instance_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            instance_loss = F.cross_entropy(
                max_instance_logits, target
            )

            loss = (
                0.5 * bag_loss
                + 0.5 * instance_loss
            )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files,
                desc=f"Epoch {epoch} Val",
            ):
                features, edges, label = get_data(
                    os.path.join(train_data_path, filename),
                    device,
                )

                _, bag_logits, _, _ = model(
                    features, edges
                )

                # 按包级验证CE选择最佳检查点
                loss = F.cross_entropy(
                    bag_logits,
                    label.long().reshape(-1),
                )
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))

        saved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            remaining = patience

            torch.save(
                model,
                os.path.join(save_path, "model.pth"),
            )
            saved = "saved"

        # 与刚才DSMIL代码保持相同的早停计数方式
        remaining -= 1

        info = (
            f"epoch:{epoch}\t"
            f"remaining:{remaining}\t"
            f"train:{train_loss:.6f}\t"
            f"val:{val_loss:.6f}\t{saved}"
        )
        logger.info(info)
        print(info)

        if remaining < 1:
            logger.info("Early stopping")
            print("Early stopping")
            break

    logger.info(
        "Best epoch: %s | Best validation CE: %.6f",
        best_epoch, best_val_loss,
    )
    print("最佳轮次：", best_epoch)
    print("最佳验证CE：", best_val_loss)
    print("权重位置：", os.path.abspath(
        os.path.join(save_path, "model.pth")
    ))

finally:
    logger.removeHandler(handler)
    handler.close()

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# 先执行MyModel及其依赖类、get_data的定义
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

save_path = "./model/MIL/temp/ours/"
test_data_path = (
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

test_files = sorted(
    filename
    for filename in os.listdir(test_data_path)
    if filename.endswith(".pth")
)
assert len(test_files) == 129, (
    f"预期129张测试切片，实际{len(test_files)}张"
)

model = torch.load(
    os.path.join(save_path, "model.pth"),
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

labels = []
predictions = []

with torch.no_grad():
    for filename in tqdm(test_files, desc="Ours Test"):
        features, edges, label = get_data(
            os.path.join(test_data_path, filename),
            device,
        )

        # 本文模型使用原始空间边
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        # 仅使用包级输出预测
        prediction = int(
            bag_logits.argmax(dim=1).item()
        )
        labels.append(int(label.item()))
        predictions.append(prediction)

assert [labels.count(0), labels.count(1)] == [81, 48], (
    "测试集标签分布与此前统计不一致"
)

summary = (
    "TEST CAMELYON16_Ours_Joint_Patience5 | "
    f"Pre: {precision_score(labels, predictions, pos_label=1, zero_division=0):.3f}   "
    f"Acc: {accuracy_score(labels, predictions):.3f}   "
    f"Rec: {recall_score(labels, predictions, pos_label=1, zero_division=0):.3f}   "
    f"F1: {f1_score(labels, predictions, pos_label=1, zero_division=0):.3f}"
)

cm = confusion_matrix(
    labels, predictions, labels=[0, 1]
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别")
print(cm)

# 追加记录，不覆盖训练日志
with open(
    os.path.join(save_path, "log.log"),
    "a",
    encoding="utf-8",
) as f:
    f.write("\n" + summary + "\n")
    f.write("Confusion matrix [0, 1]:\n")
    f.write(np.array2string(cm) + "\n")

In [ ]:
device = torch.device("cuda:0")

test_data_path = "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
test_data_files = np.array(os.listdir(test_data_path))

# model = torch.load("./model/MIL/Ablation/MIL_20/model.pth",weights_only=False)
model = torch.load("./model/MIL/Ablation/MIL_20/model.pth",weights_only=False)

model.eval()
pred_list = []
label_list = []
model.eval()
for f in tqdm(test_data_files[:]):
    with torch.no_grad():
        features,edges,label = get_data(os.path.join(test_data_path,f),device)

        prediction_i, prediction_b, attscore, feature_b = model(features,edges)
        prediction_b = torch.sigmoid(prediction_b).cpu().numpy()

        label_list.append(label.item())
        pred_list.append(prediction_b)
label = np.array(label_list)
pred = np.array(pred_list)
# pred = pred.argmax(-1)
pred = pred.argmax(-1).reshape(-1)
print(f"Pre: {precision_score(label,pred):.3f}   Acc: {accuracy_score(label,pred):.3f}   Rec: {recall_score(label,pred):.3f}   F1: {f1_score(label,pred):.3f}")
# print(f"Pre: {precision_score(label,pred,average='macro'):.3f}   Acc: {accuracy_score(label,pred):.3f}   Rec: {recall_score(label,pred,average='macro'):.3f}   F1: {f1_score(label,pred,average='macro'):.3f}")

In [ ]:
# Camelyon
device = torch.device("cuda:0")
model = torch.load("./model/MIL/Ablation/MIL_20/model.pth",weights_only=False)
path = "../../CAMELYON16/pkl_data/training/20xbags/tumor_037_7575_1.pth"

features,edges,label = get_data(path,device)
data = torch.load(path)
rects = data["rects"]
level = data["level"]
level_base = data["level_base"]
p_size = data["patch_size"]

wsi = WSI("../../CAMELYON16/training/tumor/tumor_037.tif")
mask=None
mask,_ = wsi.get_mask("../../CAMELYON16/label/training/tumor_037.xml",level_base)


prediction_i, prediction_b, attscore, feature_b = model(features,edges)
prediction_b = torch.sigmoid(prediction_b)[:, -1]



p = 2**(level_base-level)
w_size = p_size//p
img = wsi.get_image(level_base)
mask_pred = np.zeros((img.shape[0],img.shape[1]))
if mask is None:
    mask = np.zeros((img.shape[0],img.shape[1]))

# attscore = attscore[0]
att = (attscore-attscore.min())/(attscore.max()-attscore.min())

for (x,y),v in zip(rects,att):
    x,y = x//p,y//p
    cv.rectangle(img,(y,x),(y+w_size,x+w_size),(0,0,0))
    mask_pred[x:x+w_size,y:y+w_size] = v[1].item()
    # mask_pred[x:x+w_size,y:y+w_size] = v.item()

    # break

print(label,prediction_b)

plt.figure(figsize=(18,12))
plt.subplot(1,3,1)
plt.axis("off")
plt.imshow(img)

plt.subplot(1,3,2)
plt.axis("off")
plt.imshow(mask)

# mask_pred[mask_pred>mask_pred.max()*0.5]=1
# mask_pred[mask_pred<1]=0
plt.subplot(1,3,3)
plt.axis("off")
plt.imshow(mask_pred,cmap="coolwarm")
plt.show()

rec,iou = ((mask_pred*mask)>0).sum()/((mask)>0).sum(), ((mask_pred*mask)>0).sum()/((mask_pred+mask)>0).sum()
iou,rec

In [ ]:
# # BRACS
# device = torch.device("cuda:0")
# model = torch.load("./model/MIL/BRACS_New/model.pth")
# path = "../../BRACS/pkl_data/testing/BRACS_1945_1534_2.pth"

# features,edges,label = get_data(path,device)
# data = torch.load(path)
# rects = data["rects"]
# level = data["level"]
# level_base = data["level_base"]
# p_size = data["patch_size"]

# # wsi = WSI("../../BACH/WSI/train/wsi/A01.svs")
# wsi = WSI("D:/TJL/testing/Insitu/BRACS_1945.svs")


# prediction_i, prediction_b, attscore, feature_b = model(features,edges)
# prediction_b = torch.sigmoid(prediction_b)



# p = 4**(level_base-level)
# w_size = p_size//p
# img = wsi.get_image(level_base)
# mask_pred = np.zeros((img.shape[0],img.shape[1]))

# # attscore = attscore[0]
# att = (attscore-attscore.min())/(attscore.max()-attscore.min())

# for (x,y),v in zip(rects,att):
#     x,y = x//p,y//p
#     cv.rectangle(img,(y,x),(y+w_size,x+w_size),(0,0,0))
#     mask_pred[x:x+w_size,y:y+w_size] = v[2].item()

# print(label,prediction_b,prediction_b.argmax())

# plt.figure(figsize=(18,12))
# plt.subplot(1,2,1)
# plt.axis("off")
# plt.imshow(img)

# plt.subplot(1,2,2)
# plt.axis("off")
# plt.imshow(mask_pred,cmap="coolwarm")
# plt.show()

# wsi.level_dimensions,level,level_base


In [ ]:
# path = "D:/TJL/testing/Normal/"
# for f in os.listdir(path):

#     wsi = WSI(os.path.join(path,f))
#     # wsi = WSI("../../CAMELYON16/training/normal/normal_002.tif")
#     print(wsi.level_downsamples)


In [ ]:
# wsi = WSI("D:/TJL/testing/Benign/BRACS_1248.svs")
# wsi